In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats, signal
from scipy.fft import fft, fftfreq
from scipy.stats import skew, kurtosis, entropy
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.ensemble import RandomForestClassifier, VotingClassifier, GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score, precision_score
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.decomposition import PCA
from sklearn.feature_selection import SelectKBest, f_classif, RFE
from sklearn.impute import SimpleImputer
import xgboost as xgb
from lightgbm import LGBMClassifier
from imblearn.over_sampling import SMOTE, ADASYN
from imblearn.combine import SMOTETomek
import warnings
warnings.filterwarnings('ignore')

print("="*60)
print("🚀 پروژه تشخیص خرابی بیرینگ - نسخه نهایی برتر (رفع خطا)")
print("="*60)

# ============================================================
# فاز ۱: پاک‌سازی و پیش‌پردازش
# ============================================================
print("\n" + "="*60)
print("📊 فاز ۱: پاک‌سازی و پیش‌پردازش پیشرفته")
print("="*60)

df = pd.read_csv("DataSetbearing-failure.csv")
print(f"\n📊 داده خام: {df.shape}")

# بررسی NaN
print(f"\n🔍 بررسی NaN:")
print(df.isnull().sum())

# حذف NaN
df = df.dropna()
print(f"✅ بعد از حذف NaN: {df.shape}")

# ===== مدیریت Outlier با روش ایمن =====
features = [col for col in df.columns if col != 'Label']
X = df[features].copy()
y = df['Label'].copy()

# Winsorizing با IQR
for col in X.columns:
    Q1 = X[col].quantile(0.05)
    Q3 = X[col].quantile(0.95)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    X[col] = X[col].clip(lower, upper)

# حذف Outlier با Z-Score (با مدیریت ایندکس)
try:
    from scipy import stats
    z_scores = np.abs(stats.zscore(X))
    outlier_mask = (z_scores > 3).any(axis=1)
    
    # تبدیل به numpy array برای جلوگیری از مشکل ایندکس
    outlier_mask_array = np.array(outlier_mask)
    clean_indices = ~outlier_mask_array
    
    X_clean = X.iloc[clean_indices].copy()
    y_clean = y.iloc[clean_indices].copy()
    
    print(f"✅ نمونه اولیه: {len(df)} → نهایی: {len(X_clean)}")
    print(f"✅ درصد حذف: {(1 - len(X_clean)/len(df))*100:.1f}%")
except Exception as e:
    print(f"⚠️ خطا در حذف Outlier: {e}")
    X_clean = X.copy()
    y_clean = y.copy()

df_clean = X_clean.copy()
df_clean['Label'] = y_clean.values

# ============================================================
# فاز ۲: مهندسی ویژگی‌های پیشرفته
# ============================================================
print("\n" + "="*60)
print("🔬 فاز ۲: مهندسی ویژگی‌های پیشرفته")
print("="*60)

df = df_clean.copy()
eps = 1e-9

print("\n✅ افزودن ویژگی‌های جدید:")

# ۱. نسبت‌ها
df["peak_to_rms"] = df["Vel, Peak (RMS)"] / (df["Vel, Rms (RMS)"] + eps)
df["pp_to_rms"] = df["Vel, Peak to peak (RMS)"] / (df["Vel, Rms (RMS)"] + eps)
df["pp_to_peak"] = df["Vel, Peak to peak (RMS)"] / (df["Vel, Peak (RMS)"] + eps)
df["acc_vel_ratio"] = df["Acc, Rms (RMS)"] / (df["Vel, Rms (RMS)"] + eps)

# ۲. شاخص‌های خرابی
df["crest_factor"] = df["Vel, Peak (RMS)"] / (df["Vel, Rms (RMS)"] + eps)
df["impulse_factor"] = df["Vel, Peak (RMS)"] / (np.abs(df["Vel, Rms (RMS)"]) + eps)
df["margin_factor"] = df["Vel, Peak (RMS)"] / ((np.square(np.sqrt(np.abs(df["Vel, Rms (RMS)"])))) + eps)
df["kurtosis_index"] = df["Kurt (RMS)"] / 3
df["severity_index"] = df["Vel, Rms (RMS)"] * df["Acc, Rms (RMS)"] * df["Crest (RMS)"]
df["early_fault_index"] = (df["Kurt (RMS)"] * df["Crest (RMS)"]) / (df["Acc, Rms (RMS)"] + eps)
df["composite_fault"] = ((df["Kurt (RMS)"] / 3) * (df["Crest (RMS)"] / (df["Crest (RMS)"].mean() + eps)) * (df["Vel, Peak (RMS)"] / (df["Vel, Rms (RMS)"] + eps)))

# ۳. تبدیلات
for col in ["Vel, Rms (RMS)", "Acc, Rms (RMS)", "Kurt (RMS)", "Crest (RMS)"]:
    df[f"{col}_log"] = np.log1p(np.maximum(0, df[col]))
    df[f"{col}_sq"] = df[col] ** 2

# ۴. تعاملات
df["kurt_crest"] = df["Kurt (RMS)"] * df["Crest (RMS)"]
df["kurt_vel"] = df["Kurt (RMS)"] * df["Vel, Rms (RMS)"]
df["vel_acc"] = df["Vel, Rms (RMS)"] * df["Acc, Rms (RMS)"]

print(f"✅ تعداد کل ویژگی‌ها: {len(df.columns) - 1}")

# ============================================================
# فاز ۳: تقسیم‌بندی و متعادل‌سازی
# ============================================================
print("\n" + "="*60)
print("⚖️ فاز ۳: تقسیم‌بندی و متعادل‌سازی")
print("="*60)

X = df.drop(columns=["Label"]).copy()
y = df["Label"].copy()

# بررسی نهایی NaN
print(f"\n🔍 بررسی نهایی NaN:")
print(f"  NaN در X: {X.isnull().sum().sum()}")
print(f"  NaN در y: {pd.isnull(y).sum()}")

if X.isnull().sum().sum() > 0:
    imputer = SimpleImputer(strategy='median')
    X = pd.DataFrame(imputer.fit_transform(X), columns=X.columns)
    print("✅ NaN در X پر شد")

if pd.isnull(y).sum() > 0:
    clean_mask = ~pd.isnull(y)
    X = X.loc[clean_mask]
    y = y.loc[clean_mask]
    print(f"✅ NaN در y حذف شد")

# ===== تقسیم‌بندی =====
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=42, stratify=y_temp
)

print(f"\nتوزیع کلاس‌ها:")
print(f"  Train: {dict(pd.Series(y_train).value_counts().sort_index())}")
print(f"  Val: {dict(pd.Series(y_val).value_counts().sort_index())}")
print(f"  Test: {dict(pd.Series(y_test).value_counts().sort_index())}")

# ===== SMOTE =====
print("\n✅ اعمال SMOTE...")
try:
    # اطمینان از اینکه داده‌ها numpy array هستند
    X_train_array = np.array(X_train)
    y_train_array = np.array(y_train)
    
    smote = SMOTE(random_state=42, k_neighbors=5)
    X_train_resampled, y_train_resampled = smote.fit_resample(X_train_array, y_train_array)
    
    # تبدیل به DataFrame برای حفظ نام ستون‌ها
    X_train_resampled = pd.DataFrame(X_train_resampled, columns=X_train.columns)
    y_train_resampled = pd.Series(y_train_resampled)
    
    print(f"  بعد از SMOTE: {dict(pd.Series(y_train_resampled).value_counts().sort_index())}")
except Exception as e:
    print(f"  ⚠️ SMOTE ناموفق: {e}")
    X_train_resampled = X_train.copy()
    y_train_resampled = y_train.copy()

# ===== Feature Selection با RFE =====
print("\n✅ Feature Selection با RFE...")
try:
    rf_selector = RandomForestClassifier(n_estimators=100, random_state=42)
    X_train_array = np.array(X_train_resampled)
    y_train_array = np.array(y_train_resampled)
    
    rfe = RFE(estimator=rf_selector, n_features_to_select=30)
    X_train_selected = rfe.fit_transform(X_train_array, y_train_array)
    
    X_val_array = np.array(X_val)
    X_val_selected = rfe.transform(X_val_array)
    
    X_test_array = np.array(X_test)
    X_test_selected = rfe.transform(X_test_array)
    
    selected_features = X.columns[rfe.get_support_()].tolist()
    print(f"  تعداد ویژگی‌های انتخاب‌شده: {len(selected_features)}")
except Exception as e:
    print(f"  ⚠️ RFE ناموفق: {e}")
    # Fallback به SelectKBest
    try:
        selector = SelectKBest(f_classif, k=min(30, X_train_resampled.shape[1]))
        X_train_selected = selector.fit_transform(X_train_resampled, y_train_resampled)
        X_val_selected = selector.transform(X_val)
        X_test_selected = selector.transform(X_test)
        selected_features = X.columns[selector.get_support()].tolist()
        print(f"  تعداد ویژگی‌های انتخاب‌شده: {len(selected_features)}")
    except:
        # اگر همه چیز ناموفق بود، از همه ویژگی‌ها استفاده کن
        X_train_selected = np.array(X_train_resampled)
        X_val_selected = np.array(X_val)
        X_test_selected = np.array(X_test)
        selected_features = X.columns.tolist()
        print(f"  استفاده از همه {len(selected_features)} ویژگی")

# ===== Scaling =====
print("\n✅ Scaling با RobustScaler...")
scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train_selected)
X_val_scaled = scaler.transform(X_val_selected)
X_test_scaled = scaler.transform(X_test_selected)

# ============================================================
# فاز ۴: Hyperparameter Tuning
# ============================================================
print("\n" + "="*60)
print("🔧 فاز ۴: Hyperparameter Tuning")
print("="*60)

print("\n✅ تنظیم پارامترها با GridSearch...")

param_grid = {
    'max_depth': [4, 6, 8],
    'learning_rate': [0.03, 0.05, 0.07],
    'n_estimators': [200, 300, 500],
    'subsample': [0.7, 0.8, 0.9],
    'colsample_bytree': [0.7, 0.8, 0.9],
}

try:
    xgb_model = xgb.XGBClassifier(random_state=42, eval_metric="logloss", verbosity=0)
    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    grid_search = GridSearchCV(
        xgb_model, param_grid, cv=cv,
        scoring='f1_weighted', n_jobs=-1, verbose=0
    )
    grid_search.fit(X_train_scaled, y_train_resampled)
    
    best_params = grid_search.best_params_
    print(f"  ✅ بهترین پارامترها: {best_params}")
    print(f"  ✅ بهترین امتیاز: {grid_search.best_score_:.4f}")
except Exception as e:
    print(f"  ⚠️ GridSearch ناموفق: {e}")
    best_params = {
        'max_depth': 6,
        'learning_rate': 0.05,
        'n_estimators': 300,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
    }
    print(f"  استفاده از پارامترهای پیش‌فرض: {best_params}")

# ============================================================
# فاز ۵: آموزش مدل نهایی با ۶ مدل
# ============================================================
print("\n" + "="*60)
print("🚀 فاز ۵: آموزش مدل نهایی با ۶ مدل")
print("="*60)

class UltimateClassifierV7:
    def __init__(self):
        self.models = {}
        self.ensemble = None
        self.thresholds = {0: 0.25, 1: 0.25, 2: 0.22}
        self.class_weights = None
        
    def compute_weights(self, y):
        """محاسبه وزن‌های هوشمند"""
        class_counts = np.bincount(y)
        total = len(y)
        n_classes = len(class_counts)
        
        weights = {}
        for i, count in enumerate(class_counts):
            weights[i] = (total / (n_classes * count + 1)) ** 0.7
            if i == 2:
                weights[i] *= 1.4
            elif i == 1:
                weights[i] *= 1.2
        
        max_w = max(weights.values())
        weights = {i: w / max_w for i, w in weights.items()}
        return weights
    
    def fit(self, X, y, X_val=None, y_val=None):
        """آموزش کامل مدل با ۶ مدل"""
        
        self.class_weights = self.compute_weights(y)
        print(f"\n✅ وزن‌های کلاس‌ها: {self.class_weights}")
        
        # ۱. XGBoost اصلی
        print("\n✅ آموزش XGBoost 1...", end=" ")
        xgb1 = xgb.XGBClassifier(
            **best_params,
            random_state=42,
            eval_metric="logloss",
            verbosity=0
        )
        xgb1.fit(X, y)
        self.models['xgb1'] = xgb1
        print("✓")
        
        # ۲. XGBoost با وزن بیشتر برای کلاس ۲
        print("✅ آموزش XGBoost 2 (وزن کلاس ۲)...", end=" ")
        xgb2 = xgb.XGBClassifier(
            **best_params,
            scale_pos_weight=self.class_weights[2] * 1.5,
            random_state=123,
            eval_metric="logloss",
            verbosity=0
        )
        xgb2.fit(X, y)
        self.models['xgb2'] = xgb2
        print("✓")
        
        # ۳. LightGBM
        print("✅ آموزش LightGBM...", end=" ")
        lgbm = LGBMClassifier(
            n_estimators=300,
            learning_rate=0.05,
            num_leaves=31,
            max_depth=8,
            class_weight='balanced',
            random_state=42,
            verbose=-1
        )
        lgbm.fit(X, y)
        self.models['lgbm'] = lgbm
        print("✓")
        
        # ۴. RandomForest
        print("✅ آموزش RandomForest...", end=" ")
        rf = RandomForestClassifier(
            n_estimators=300,
            max_depth=10,
            class_weight='balanced_subsample',
            random_state=42
        )
        rf.fit(X, y)
        self.models['rf'] = rf
        print("✓")
        
        # ۵. GradientBoosting
        print("✅ آموزش GradientBoosting...", end=" ")
        gb = GradientBoostingClassifier(
            n_estimators=200,
            learning_rate=0.05,
            max_depth=6,
            random_state=42
        )
        gb.fit(X, y)
        self.models['gb'] = gb
        print("✓")
        
        # ۶. XGBoost با تنظیمات متفاوت
        print("✅ آموزش XGBoost 3 (تنظیمات متفاوت)...", end=" ")
        xgb3 = xgb.XGBClassifier(
            max_depth=8,
            learning_rate=0.07,
            n_estimators=200,
            subsample=0.7,
            colsample_bytree=0.7,
            random_state=456,
            eval_metric="logloss",
            verbosity=0
        )
        xgb3.fit(X, y)
        self.models['xgb3'] = xgb3
        print("✓")
        
        # ===== Ensemble =====
        print("\n✅ ایجاد Ensemble با ۶ مدل...", end=" ")
        estimators = [
            ('xgb1', xgb1),
            ('xgb2', xgb2),
            ('lgbm', lgbm),
            ('rf', rf),
            ('gb', gb),
            ('xgb3', xgb3)
        ]
        
        self.ensemble = VotingClassifier(
            estimators=estimators,
            voting='soft',
            weights=[1.2, 1.3, 1.0, 0.8, 0.7, 1.1]
        )
        self.ensemble.fit(X, y)
        print("✓")
        
        # ===== تنظیم آستانه‌ها =====
        if X_val is not None and y_val is not None:
            print("\n✅ تنظیم آستانه‌های بهینه...", end=" ")
            probs_val = self.ensemble.predict_proba(X_val)
            
            best_thresholds = self.thresholds.copy()
            best_f1 = 0
            
            for th0 in np.arange(0.15, 0.45, 0.03):
                for th1 in np.arange(0.15, 0.45, 0.03):
                    for th2 in np.arange(0.15, 0.40, 0.03):
                        test_th = {0: th0, 1: th1, 2: th2}
                        preds = self._apply_thresholds(probs_val, test_th)
                        try:
                            f1 = f1_score(y_val, preds, average='weighted')
                            if f1 > best_f1:
                                best_f1 = f1
                                best_thresholds = test_th.copy()
                        except:
                            pass
            
            self.thresholds = best_thresholds
            print(f"✓")
            print(f"   آستانه‌ها: {self.thresholds}")
            print(f"   بهترین F1: {best_f1:.4f}")
        
        return self
    
    def _apply_thresholds(self, probs, thresholds):
        preds = np.argmax(probs, axis=1)
        for class_label, thresh in thresholds.items():
            mask = probs[:, class_label] >= thresh
            preds[mask] = class_label
        return preds
    
    def predict(self, X):
        probs = self.ensemble.predict_proba(X)
        preds = self._apply_thresholds(probs, self.thresholds)
        
        if len(np.unique(preds)) < 3:
            preds = np.argmax(probs, axis=1)
        
        return preds
    
    def predict_proba(self, X):
        return self.ensemble.predict_proba(X)
    
    def evaluate(self, X, y):
        preds = self.predict(X)
        
        print("\n" + "="*60)
        print("📊 گزارش عملکرد نهایی:")
        print("="*60)
        print(classification_report(y, preds, 
              target_names=["کلاس ۰ (سالم)", "کلاس ۱ (خرابی)", "کلاس ۲ (خرابی خفیف)"]))
        
        cm = confusion_matrix(y, preds)
        print("\n📊 ماتریس درهم‌ریختگی:")
        print("                 پیش‌بینی")
        print("              کلاس0  کلاس1  کلاس2")
        for i in range(3):
            print(f"واقعی کلاس{i}   {cm[i,0]:6d}  {cm[i,1]:6d}  {cm[i,2]:6d}")
        
        accuracy = accuracy_score(y, preds)
        f1 = f1_score(y, preds, average='weighted')
        f1_per_class = f1_score(y, preds, average=None)
        
        print(f"\n📊 معیارهای نهایی:")
        print(f"   Accuracy: {accuracy:.4f}")
        print(f"   F1-Score (Weighted): {f1:.4f}")
        print(f"   F1-Score per class: {f1_per_class}")
        
        return {
            'accuracy': accuracy,
            'f1_weighted': f1,
            'f1_per_class': f1_per_class,
            'confusion_matrix': cm,
            'predictions': preds
        }

# ============================================================
# اجرای مدل نهایی
# ============================================================
print("\n" + "="*60)
print("🚀 شروع آموزش مدل نهایی برتر")
print("="*60)

model = UltimateClassifierV7()
model.fit(X_train_scaled, y_train_resampled, X_val_scaled, y_val)

# ارزیابی
results = model.evaluate(X_test_scaled, y_test)

# ============================================================
# ذخیره نتایج
# ============================================================
print("\n" + "="*60)
print("💾 ذخیره نتایج نهایی")
print("="*60)

with open('model_results_best_fixed.txt', 'w', encoding='utf-8') as f:
    f.write("="*60 + "\n")
    f.write("گزارش نهایی مدل تشخیص خرابی بیرینگ (نسخه برتر)\n")
    f.write("="*60 + "\n\n")
    
    f.write("بهترین پارامترهای XGBoost:\n")
    for param, value in best_params.items():
        f.write(f"  {param}: {value}\n")
    
    f.write(f"\nوزن‌های کلاس‌ها:\n")
    for i, w in model.class_weights.items():
        f.write(f"  کلاس {i}: {w:.3f}\n")
    
    f.write(f"\nآستانه‌های بهینه:\n")
    for i, th in model.thresholds.items():
        f.write(f"  کلاس {i}: {th:.2f}\n")
    
    f.write(f"\nتعداد مدل‌های Ensemble: 6\n")
    f.write(f"تعداد ویژگی‌ها: {len(selected_features)}\n\n")
    
    f.write("عملکرد روی داده Test:\n")
    f.write(classification_report(
        y_test, 
        model.predict(X_test_scaled), 
        target_names=['سالم', 'خرابی', 'خرابی خفیف']
    ))
    
    f.write("\nماتریس درهم‌ریختگی:\n")
    f.write(str(results['confusion_matrix']) + "\n")
    
    f.write(f"\nمعیارهای نهایی:\n")
    f.write(f"  Accuracy: {results['accuracy']:.4f}\n")
    f.write(f"  F1-Score (Weighted): {results['f1_weighted']:.4f}\n")
    f.write(f"  F1-Score per class: {results['f1_per_class']}\n")

print("\n✅ نتایج ذخیره شد: model_results_best_fixed.txt")

print("\n" + "="*60)
print("📊 خلاصه نهایی:")
print("="*60)
print(f"   ✅ Accuracy: {results['accuracy']:.4f}")
print(f"   ✅ F1-Score (Weighted): {results['f1_weighted']:.4f}")
print(f"   ✅ F1-Score کلاس ۰: {results['f1_per_class'][0]:.4f}")
print(f"   ✅ F1-Score کلاس ۱: {results['f1_per_class'][1]:.4f}")
print(f"   ✅ F1-Score کلاس ۲: {results['f1_per_class'][2]:.4f}")
print(f"   ✅ تعداد مدل‌ها: 6")
print(f"   ✅ تعداد ویژگی‌ها: {len(selected_features)}")

print("\n🎉 پروژه با موفقیت کامل به پایان رسید!")